In [ ]:
import pandas as pd
import numpy as np

def generate_synthetic_data(N=10000, random_seed=42):
    """
    Generates a synthetic dataset for Mobility as a Service (MaaS) choice modeling.
    Includes non-linear latent variable structures and multiple choice alternatives.
    """
    np.random.seed(random_seed)
    
    # 1. Generate Socio-demographics (Z: Age and Income)
    # Sampling from uniform distributions within realistic ranges
    age = np.random.uniform(18, 70, N)
    income = np.random.uniform(20, 150, N)
    
    # Standardization for modeling consistency
    age_norm = (age - np.mean(age)) / np.std(age)
    income_norm = (income - np.mean(income)) / np.std(income)
    
    # 2. Define Ground-Truth Latent Variable (LV)
    # The LV incorporates strong non-linear effects: 
    true_lv = -1.2 * (age_norm ** 2) + 1.2 * np.tanh(income_norm) + np.random.normal(0, 1, N)
    
    # 3. Measurement Equations (Indicator generation for 1-5 Likert scale)
    # Latent constructs are mapped to continuous indicator stars with random noise
    ind1_star = 1.2 * true_lv + np.random.normal(0, 1, N)
    ind2_star = 0.9 * true_lv + np.random.normal(0, 1, N)
    ind3_star = 1.0 * true_lv + np.random.normal(0, 1, N)
    
    def discretize_to_likert(ind_star):
        """Maps continuous latent responses to discrete 5-point Likert scales."""
        bins = np.percentile(ind_star, [15, 35, 65, 85])
        return np.digitize(ind_star, bins) + 1

    ind1 = discretize_to_likert(ind1_star)
    ind2 = discretize_to_likert(ind2_star)
    ind3 = discretize_to_likert(ind3_star)
    
    # 4. Generate Alternative-Specific Attributes (X: Time and Cost)
    # Car: Normal commute range
    time_car = np.random.normal(30, 10, N).clip(10, 60)
    cost_car = np.random.normal(15, 5, N).clip(5, 30)
    
    # Public Transit (PT): Generally longer time, lower cost
    time_pt = np.random.normal(45, 15, N).clip(20, 80)
    cost_pt = np.random.normal(5, 2, N).clip(1, 15)
    
    # MaaS: Competitive time/cost with behavioral influence from LV
    time_maas = np.random.normal(35, 12, N).clip(15, 70)
    cost_maas = np.random.normal(12, 4, N).clip(3, 25)
    
    # 5. Define Systematic Utility Parameters
    # Parameters are set to ensure significant differentiation between alternatives
    beta_time = -0.08  # Travel time penalty
    beta_cost = -0.20  # Travel cost penalty
    gamma_lv = 2.0     # Influence of Pro-MaaS attitude on choice utility
    
    asc_car = 0.0      # Alternative Specific Constant (Reference: Car)
    asc_pt = -0.5
    asc_maas = 0.5
    
    # Utility Function Calculations
    v_car = asc_car + beta_time * time_car + beta_cost * cost_car
    v_pt = asc_pt + beta_time * time_pt + beta_cost * cost_pt
    v_maas = asc_maas + beta_time * time_maas + beta_cost * cost_maas + gamma_lv * true_lv
    
    # 6. Choice Generation (Multinomial Logit Kernel)
    exp_car = np.exp(v_car)
    exp_pt = np.exp(v_pt)
    exp_maas = np.exp(v_maas)
    sum_exp = exp_car + exp_pt + exp_maas
    
    prob_car = exp_car / sum_exp
    prob_pt = exp_pt / sum_exp
    prob_maas = exp_maas / sum_exp
    
    # Stochastic choice simulation
    r = np.random.rand(N)
    choice = np.zeros(N, dtype=int)
    choice[r > prob_car] = 1
    choice[r > (prob_car + prob_pt)] = 2
    
    # 7. Model Diagnostic: Deterministic Error Rate
    # Calculates the percentage of choices that deviate from the maximum systematic utility
    v_all = np.column_stack((v_car, v_pt, v_maas))
    deterministic_choice = np.argmax(v_all, axis=1)
    error_rate = np.mean(deterministic_choice != choice)
    print(f"Synthetic Dataset Diagnostic - Deterministic Error Rate: {error_rate * 100:.2f}%")
    
    # 8. Data Export
    df = pd.DataFrame({
        'Age': age, 'Income': income,
        'Time_Car': time_car, 'Cost_Car': cost_car,
        'Time_PT': time_pt, 'Cost_PT': cost_pt,
        'Time_MaaS': time_maas, 'Cost_MaaS': cost_maas,
        'Ind_1': ind1, 'Ind_2': ind2, 'Ind_3': ind3,
        'True_LV': true_lv,
        'Choice': choice
    })
    
    df.to_csv('synthetic_maas_dataset.csv', index=False)
    print("Synthetic dataset successfully generated and saved to 'synthetic_maas_dataset.csv'.")
    return df

if __name__ == "__main__":
    generate_synthetic_data()